<a href="https://colab.research.google.com/github/SiriBoge/BDA_ASSIGNMENT02_092/blob/main/BDA_ASSIGNMENT02_092.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1.Build a Classification Model with Spark with a dataset of your choice**

Used the Iris dataset (via sklearn) converted into a Spark DataFrame. Applied VectorAssembler to combine all 4 features into a single feature vector. Trained a Random Forest Classifier with 100 trees on an 80/20 train-test split. Evaluated using accuracy score.

In [1]:
# Run this first cell to install PySpark
!pip install pyspark

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Classification").getOrCreate()

In [2]:
# ── 1. Install & Start Spark ──────────────────────────────────────


from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("ClassificationModel").getOrCreate()

# ── 2. Load Dataset (Iris - built-in via sklearn) ─────────────────
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
df_pd = pd.DataFrame(iris.data, columns=["sepal_length","sepal_width","petal_length","petal_width"])
df_pd["label"] = iris.target

# Convert to Spark DataFrame
df = spark.createDataFrame(df_pd)
df.show(5)

# ── 3. Feature Engineering ────────────────────────────────────────
assembler = VectorAssembler(
    inputCols=["sepal_length","sepal_width","petal_length","petal_width"],
    outputCol="features"
)
df = assembler.transform(df)

# ── 4. Train/Test Split ───────────────────────────────────────────
train, test = df.randomSplit([0.8, 0.2], seed=42)

# ── 5. Train Model ────────────────────────────────────────────────
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100)
model = rf.fit(train)

# ── 6. Evaluate ───────────────────────────────────────────────────
predictions = model.transform(test)
evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"✅ Accuracy: {accuracy:.4f}")

predictions.select("label", "prediction", "probability").show(10)

+------------+-----------+------------+-----------+-----+
|sepal_length|sepal_width|petal_length|petal_width|label|
+------------+-----------+------------+-----------+-----+
|         5.1|        3.5|         1.4|        0.2|    0|
|         4.9|        3.0|         1.4|        0.2|    0|
|         4.7|        3.2|         1.3|        0.2|    0|
|         4.6|        3.1|         1.5|        0.2|    0|
|         5.0|        3.6|         1.4|        0.2|    0|
+------------+-----------+------------+-----------+-----+
only showing top 5 rows
✅ Accuracy: 0.9688
+-----+----------+---------------+
|label|prediction|    probability|
+-----+----------+---------------+
|    0|       0.0|  [1.0,0.0,0.0]|
|    0|       0.0|  [1.0,0.0,0.0]|
|    0|       0.0|  [1.0,0.0,0.0]|
|    0|       0.0|[0.98,0.02,0.0]|
|    0|       0.0|  [1.0,0.0,0.0]|
|    0|       0.0|  [1.0,0.0,0.0]|
|    0|       0.0|  [1.0,0.0,0.0]|
|    0|       0.0|  [1.0,0.0,0.0]|
|    0|       0.0|  [1.0,0.0,0.0]|
|    0|       0

**2.Build a Clustering Model with Spark with a dataset of your choice**

Used the same Iris dataset but without labels (unsupervised). Applied VectorAssembler + StandardScaler to normalize features. Trained a KMeans model with k=3 (since we know there are 3 flower species). Evaluated using Silhouette Score — closer to 1 means better clustering.

In [3]:
# ── 1. Install & Start Spark ──────────────────────────────────────


from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

spark = SparkSession.builder.appName("ClusteringModel").getOrCreate()

# ── 2. Load Dataset ───────────────────────────────────────────────
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
df_pd = pd.DataFrame(iris.data, columns=["sepal_length","sepal_width","petal_length","petal_width"])
df = spark.createDataFrame(df_pd)

# ── 3. Feature Engineering ────────────────────────────────────────
assembler = VectorAssembler(
    inputCols=["sepal_length","sepal_width","petal_length","petal_width"],
    outputCol="features"
)
df = assembler.transform(df)

# Scale features
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")
df = scaler.fit(df).transform(df)

# ── 4. Train KMeans ───────────────────────────────────────────────
kmeans = KMeans(featuresCol="scaled_features", k=3, seed=42)
model = kmeans.fit(df)

# ── 5. Evaluate ───────────────────────────────────────────────────
predictions = model.transform(df)
evaluator = ClusteringEvaluator(featuresCol="scaled_features")
silhouette = evaluator.evaluate(predictions)
print(f"✅ Silhouette Score: {silhouette:.4f}")

# Show cluster centers
print("\nCluster Centers:")
for i, center in enumerate(model.clusterCenters()):
    print(f"  Cluster {i}: {center}")

predictions.select("sepal_length","sepal_width","prediction").show(10)

✅ Silhouette Score: 0.6448

Cluster Centers:
  Cluster 0: [6.8887588  6.04493327 2.38782168 1.74828502]
  Cluster 1: [6.05788156 7.91761264 0.83006151 0.32128819]
  Cluster 2: [8.08674985 7.02050171 3.06927278 2.5427526 ]
+------------+-----------+----------+
|sepal_length|sepal_width|prediction|
+------------+-----------+----------+
|         5.1|        3.5|         1|
|         4.9|        3.0|         1|
|         4.7|        3.2|         1|
|         4.6|        3.1|         1|
|         5.0|        3.6|         1|
|         5.4|        3.9|         1|
|         4.6|        3.4|         1|
|         5.0|        3.4|         1|
|         4.4|        2.9|         1|
|         4.9|        3.1|         1|
+------------+-----------+----------+
only showing top 10 rows


**3.Build a Recommendation Engine with Spark with a dataset of your choice**

Created a custom movie ratings dataset (userId, movieId, rating). Trained an ALS (Alternating Least Squares) model — the standard collaborative filtering algorithm in Spark MLlib. Evaluated using RMSE (lower = better predictions). Finally generated Top 3 movie recommendations per user.

In [5]:
# ── 1. Install & Start Spark ──────────────────────────────────────


from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName("RecommendationEngine").getOrCreate()

# ── 2. Download MovieLens Dataset ─────────────────────────────────
!wget -q http://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q ml-latest-small.zip

# ── 3. Load Ratings Data ──────────────────────────────────────────
df = spark.read.csv("ml-latest-small/ratings.csv", header=True, inferSchema=True)
df = df.select("userId", "movieId", "rating")
df.show(5)
print(f"Total Ratings: {df.count()}")

# ── 4. Train/Test Split ───────────────────────────────────────────
train, test = df.randomSplit([0.8, 0.2], seed=42)

# ── 5. Train ALS Model (Tuned Parameters) ─────────────────────────
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    maxIter=15,        # more iterations
    regParam=0.05,     # reduced regularization
    rank=20,           # more latent factors
    coldStartStrategy="drop"
)
model = als.fit(train)

# ── 6. Evaluate ───────────────────────────────────────────────────
predictions = model.transform(test)
evaluator = RegressionEvaluator(
    metricName="rmse", labelCol="rating", predictionCol="prediction"
)
rmse = evaluator.evaluate(predictions)
print(f"✅ RMSE: {rmse:.4f}")

# ── 7. Top 5 Movie Recommendations per User ───────────────────────
user_recs = model.recommendForAllUsers(5)
print("\nTop 5 Movie Recommendations per User:")
user_recs.show(10, truncate=False)

# ── 8. Load Movie Titles for better display ───────────────────────
movies = spark.read.csv("ml-latest-small/movies.csv", header=True, inferSchema=True)

# Show recommendations with actual movie names for User 1
from pyspark.sql.functions import explode

user1_recs = user_recs.filter(user_recs.userId == 1) \
    .select(explode("recommendations").alias("rec")) \
    .select("rec.movieId", "rec.rating")

user1_recs.join(movies, "movieId") \
    .select("title", "rating") \
    .show(truncate=False)

+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     1|      1|   4.0|
|     1|      3|   4.0|
|     1|      6|   4.0|
|     1|     47|   5.0|
|     1|     50|   5.0|
+------+-------+------+
only showing top 5 rows
Total Ratings: 100836
✅ RMSE: 0.9423

Top 5 Movie Recommendations per User:
+------+------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                 |
+------+------------------------------------------------------------------------------------------------+
|1     |[{5013, 6.0480285}, {27611, 5.8749213}, {543, 5.8218026}, {106100, 5.7848005}, {3836, 5.723117}]|
|2     |[{2289, 4.9912763}, {1274, 4.9722505}, {131724, 4.960698}, {86377, 4.855809}, {1222, 4.845173}] |
|3     |[{6835, 4.9619207}, {5746, 4.9619207}, {5181, 4.9573417}, {4518, 4.8826137}, {2851, 4.8702655}] |
|4     |[{1035, 5.605795}, {25825, 5.325243}, {9